In [1]:
//#r "C:\Users\miao\Documents\JupyterNotebook\BoSSSpadMergeEnable\BoSSSpad.dll"
//#r "C:\Users\miao\Documents\JupyterNotebook\BoSSSpadPrintRemoveDtRestriction\BoSSSpad.dll"
//#r "C:\Users\miao\Documents\JupyterNotebook\BoSSSpadPrintRemoveDtRestrictionAMR\BoSSSpad.dll"
#r "C:\Users\miao\Documents\JupyterNotebook\BoSSSpadPrintRemoveDtRestrictionAMRDebug\BoSSSpad.dll"
//#r "C:\Users\miao\Documents\JupyterNotebook\BoSSSpadMergeNew\BoSSSpad.dll"
using System;
using System.Collections.Generic;
using System.Linq;
using System.IO;
using System.Data;
using System.Globalization;
using System.Threading;
using ilPSP;
using ilPSP.Utils;
using BoSSS.Platform;
using BoSSS.Foundation;
using BoSSS.Foundation.Grid;
using BoSSS.Foundation.Grid.Classic;
using BoSSS.Foundation.IO;
using BoSSS.Solution;
using BoSSS.Solution.Control;
using BoSSS.Solution.GridImport;
using BoSSS.Solution.Statistic;
using BoSSS.Solution.Utils;
using BoSSS.Solution.Gnuplot;
using BoSSS.Application.BoSSSpad;
using BoSSS.Application.XNSE_Solver;
using static BoSSS.Application.BoSSSpad.BoSSSshell;
using BoSSS.Foundation.Grid.RefElements;
using BoSSS.Platform.LinAlg;
using BoSSS.Solution.NSECommon;
using BoSSS.Solution.XdgTimestepping;
Init();

The below script needs to be able to find the current output cell; this is an easy method to get it.

In [2]:
using BoSSS.Foundation.Grid.Classic;
using ilPSP.Utils;
using System;
using System.Collections.Generic;
using System.Linq;
using System.Text;
using System.Threading.Tasks;
using BoSSS.Foundation.Grid;
using BoSSS.Solution.XdgTimestepping;
using BoSSS.Solution.Timestepping;
using BoSSS.Solution.Control;
using BoSSS.Solution.LevelSetTools.SolverWithLevelSetUpdater;
using ZwoLevelSetSolver;
using ZwoLevelSetSolver.SolidPhase;
using BoSSS.Solution.XNSECommon;

In [3]:
ExecutionQueues

index,type,value
0,BoSSS.Application.BoSSSpad.MiniBatchProcessorClient,MiniBatchProcessor client @C:\Users\miao\Documents\Database\Deployment
1,BoSSS.Application.BoSSSpad.MsHPC2012Client,"MS HPC client MSHPC-AllNodes @DC3, @\\dc3\userspace\miao\cluster\Deployment"
2,BoSSS.Application.BoSSSpad.MsHPC2012Client,"MS HPC client MSHPC-AllNodes @DC3, @\\dc3\userspace\miao\cluster\Deployment"


In [4]:
GetDefaultQueue()

RuntimeLocation,win\amd64
DeploymentBaseDirectory,\\dc3\userspace\miao\cluster\Deployment
DeployRuntime,True
Name,MSHPC-AllNodes
DotnetRuntime,dotnet
Username,FDY\miao
ServerName,DC3
ComputeNodes,"[ HPCCLUSTER01, HPCCLUSTER02, HPCCLUSTER03 ]"
DefaultJobPriority,Normal
SingleNode,True
AllowedDatabasesPaths,[ \\dc3\userspace\miao\cluster ]


In [5]:
static var myBatch = ExecutionQueues[2];
static var myDb = myBatch.CreateOrOpenCompatibleDatabase("Droplet-EE-LikeEL");

In [6]:
BoSSSshell.WorkflowMgm.Init("Droplet-EE-LikeEL");

Project name is set to 'Droplet-EE-LikeEL'.
Default Execution queue is chosen for the database.
Opening existing database '\\dc3\userspace\miao\cluster\Droplet-EE-LikeEL'.


In [7]:
wmg.DefaultDatabase

{ Session Count = 138; Grid Count = 836; Path = \\dc3\userspace\miao\cluster\Droplet-EE-LikeEL }

## Create grid

In [8]:
public static class GridFactory {
 
    public static Grid2D GenerateGrid() { 
        double xSize = 2;
        double yTop = 1.5 - 20.0 / 176.7;
        double yBottom = -20.0 / 176.7;
        int kelem = 8;

        double[] Xnodes = GenericBlas.Linspace(-xSize, xSize, kelem + 1);
        double[] Ynodes = GenericBlas.Linspace(yBottom, yTop, kelem / 8 * 3 + 1);
                var grd = Grid2D.Cartesian2DGrid(Xnodes, Ynodes);

                grd.EdgeTagNames.Add(1, "wall_lower");
                grd.EdgeTagNames.Add(2, "pressure_outlet_upper");
                //grd.EdgeTagNames.Add(3, "wall_left");
                //grd.EdgeTagNames.Add(4, "wall_right");
                grd.EdgeTagNames.Add(3, "FreeSlip_left");
                grd.EdgeTagNames.Add(4, "FreeSlip_right");

                grd.DefineEdgeTags(delegate (double[] X) {
                    byte et = 0;
                    if(Math.Abs(X[1] - yBottom) <= 1.0e-8)
                        et = 1;
                    if(Math.Abs(X[1] - yTop) <= 1.0e-8)
                        et = 2;
                    if(Math.Abs(X[0] + xSize) <= 1.0e-8)
                        et = 3;
                    if(Math.Abs(X[0] - xSize) <= 1.0e-8)
                        et = 4;

                    return et;
                });

                return grd;
     }
 
 }

In [9]:
public static class BoundaryValueFactory { 
    public static string GetPrefixCode() {
        using(var stw = new System.IO.StringWriter()) {
           
           stw.WriteLine("static class BoundaryValues {");
           stw.WriteLine("  static public double Zero(double[] X) {");
           stw.WriteLine("    return 0.0;");
           stw.WriteLine("  }");

           stw.WriteLine("  static public double pJump(double[] X) {");
           stw.WriteLine("    return 0.046 / 0.4;");
           stw.WriteLine("  }");
           
           stw.WriteLine(" static public double InitialPhi(double[] X) { ");
           //stw.WriteLine("    return ((X[0] - 0.0).Pow2() + (X[1] - 0.0).Pow2() - 0.16);");
            
           stw.WriteLine("    if(X[1] >= (0)) {");
           stw.WriteLine("    return ((X[0] - 0.0).Pow2() + (X[1] - 0.0).Pow2() - 0.16);");
           stw.WriteLine("    }");
           stw.WriteLine("    return ((X[0] - 0.0).Pow2() + (0.0 - 0.0).Pow2() - 0.16);");
            
           stw.WriteLine("    }"); 
           
           stw.WriteLine(" static public double InitialPhi1(double[] X) { ");
           stw.WriteLine("    return -(X[1] - 0);");
           stw.WriteLine("    }"); 

           stw.WriteLine("}"); 
           return stw.ToString();
        }
    }
   
    static public Formula Get_Zero() {
        return new Formula("BoundaryValues.Zero", AdditionalPrefixCode:GetPrefixCode());
    }

    static public Formula Get_pJump(){
        return new Formula("BoundaryValues.pJump", AdditionalPrefixCode:GetPrefixCode());
    }

    static public Formula Get_InitialPhi(){
        return new Formula("BoundaryValues.InitialPhi", AdditionalPrefixCode:GetPrefixCode());
    }
    
    static public Formula Get_InitialPhi1(){
        return new Formula("BoundaryValues.InitialPhi1", AdditionalPrefixCode:GetPrefixCode());
    }
}

## Create Control file

In [10]:
        public static ZLS_Control Aland( int p = 2, int AMRlvl = 0, int MpiNum = 1) {
            ZLS_Control C = new ZLS_Control(p);
            //C.ImmediatePlotPeriod = 1;
            //C.SuperSampling = 3;

            C.AgglomerationThreshold = 0.3;
            C.NoOfMultigridLevels = 1;

            //int D = 2;

            AppControl._TimesteppingMode compMode = AppControl._TimesteppingMode.Transient;

            //_DbPath = @"\\fdyprime\userspace\smuda\cluster\cluster_db";
            //_DbPath = @"D:\local\local_Testcase_databases\Testcase_ContactLine";
            //_DbPath = @"D:\local\local_spatialConvStudy\StaticDropletOnPlateConvergence\SDoPConvDB";

            // basic database options
            // ======================
            #region db

            C.savetodb = true;
            //C.DbPath = @"\\dc1\userspace\miao\cluster\Droplet-EE-LikeEL-DT";
            C.ProjectName = "Droplet";
            C.SessionName = "Droplet-LikeEL-AMR" + AMRlvl + "-Merged-MPI" + MpiNum + "-AMRMOD" + "Debug";
            C.ProjectDescription = "Droplet running on pc";

            C.ContinueOnIoError = false;

            //C.LogValues = XNSE_Control.LoggingValues.MovingContactLine;
            //C.PostprocessingModules.Add(new MovingContactLineLogging());

            #endregion

            // Physical Parameters
            // ===================
            #region physics

            double scale = 4.4175e-4; // For a droplet with radius r = 176.7 micrometre
            C.PhysicalParameters.rho_A = 10 * scale * scale * scale;
            C.PhysicalParameters.rho_B = 1260 * scale * scale * scale;
            C.PhysicalParameters.mu_A = 0.1 * scale ;
            C.PhysicalParameters.mu_B = 1.41 * scale;
            double sigma = 0.046;
            C.PhysicalParameters.Sigma = sigma;

            C.PhysicalParameters.betaS_A = 0.0;
            C.PhysicalParameters.betaS_B = 0.0;

            C.PhysicalParameters.betaL = 0.0;
            C.PhysicalParameters.theta_e = Math.PI / 2.0;

            C.PhysicalParameters.IncludeConvection = true;
            C.PhysicalParameters.Material = false; //??

            C.Material = new Solid() {
                Density = 1000 * scale * scale * scale,
                Lame2 = 1000 * scale,
                Viscosity = 1 * scale, // Real Viscosity
            };

            #endregion

            // grid generation
            // ===============
            #region grid

            C.SetGrid(GridFactory.GenerateGrid());

            #endregion

            // Initial Values
            // ==============

            C.AddInitialValue("Pressure#A", BoundaryValueFactory.Get_pJump());
            C.AddInitialValue("Pressure#B", BoundaryValueFactory.Get_Zero());
            C.AddInitialValue("Phi", BoundaryValueFactory.Get_InitialPhi());
            C.AddInitialValue("Phi2", BoundaryValueFactory.Get_InitialPhi1());
            //C.AddInitialValue(ZwoLevelSetSolver.VariableNames.SolidLevelSetCG, BoundaryValueFactory.Get_InitialPhi1());



            // boundary conditions
            // ===================
            #region BC


            C.AddBoundaryValue("wall_lower");
            C.AddBoundaryValue("pressure_outlet_upper");
            //C.AddBoundaryValue("wall_left");
            //C.AddBoundaryValue("wall_right");
            C.AddBoundaryValue("FreeSlip_left");
            C.AddBoundaryValue("FreeSlip_right");

            C.AdvancedDiscretizationOptions.GNBC_Localization = NavierSlip_Localization.Bulk;
            C.AdvancedDiscretizationOptions.GNBC_SlipLength = NavierSlip_SlipLength.Prescribed_Beta;
            //C.PhysicalParameters.sliplength = 0.001;

            #endregion

            // misc. solver options
            // ====================
            #region solver


            //C.AdvancedDiscretizationOptions.CellAgglomerationThreshold = 0.2;
            //C.AdvancedDiscretizationOptions.PenaltySafety = 40;
            //C.AdvancedDiscretizationOptions.UseGhostPenalties = true;

            C.NonLinearSolver.MaxSolverIterations = 160;
            C.NonLinearSolver.MinSolverIterations = 2;
            //C.Solver_MaxIterations = 50;
            C.NonLinearSolver.ConvergenceCriterion = 1e-8;
            //C.Solver_ConvergenceCriterion = 1e-8;
            C.LevelSet_ConvergenceCriterion = 1e-12;
            C.NonLinearSolver.Globalization = BoSSS.Solution.AdvancedSolvers.Newton.GlobalizationOption.Dogleg;


            //C.Option_LevelSetEvolution = (compMode == AppControl._TimesteppingMode.Steady) ? LevelSetEvolution.None : LevelSetEvolution.FastMarching;
            //C.EllipticExtVelAlgoControl.solverFactory = () => new ilPSP.LinSolvers.PARDISO.PARDISOSolver();
            //C.EllipticExtVelAlgoControl.IsotropicViscosity = 1e-3;
            //C.fullReInit = false; 

            C.AdvancedDiscretizationOptions.FilterConfiguration = CurvatureAlgorithms.FilterConfiguration.NoFilter;
            C.AdvancedDiscretizationOptions.ViscosityMode = ViscosityMode.FullySymmetric;
            C.AdvancedDiscretizationOptions.SST_isotropicMode = BoSSS.Solution.XNSECommon.SurfaceStressTensor_IsotropicMode.LaplaceBeltrami_ContactLine;

            C.AdaptiveMeshRefinement = true;
            //C.activeAMRlevelIndicators.Add(new ContactPointRefiner { maxRefinementLevel = AMRlvl});
            //C.AMR_startUpSweeps = AMRlvl;
            C.activeAMRlevelIndicators.Add(new AMRatInnerContactLine { maxRefinementLevel = AMRlvl, levelSets = new[] { 0, 1 }, FieldWidth = 1 });
            C.AMR_startUpSweeps = AMRlvl;

            #endregion
                
            C.DynamicLoadBalancing_On = true;
            C.DynamicLoadBalancing_Period = 1000;
            C.DynamicLoadBalancing_RedistributeAtStartup = true;
            C.GridPartType = GridPartType.METIS;


            // Timestepping
            // ============
            #region time

            //C.CheckJumpConditions = true;

            C.TimeSteppingScheme = TimeSteppingScheme.BDF2;
            C.Timestepper_BDFinit = TimeStepperInit.SingleInit;
            C.Timestepper_LevelSetHandling = LevelSetHandling.LieSplitting;
            

            C.TimesteppingMode = compMode;
            //double dt = 5e-7;
            double dt = 2e-5;
            C.dtMax = dt;
            C.dtMin = dt;
            C.Endtime = 100;
            C.NoOfTimesteps = 5000;
            C.saveperiod = 1;

            C.TracingNamespaces = "*";
            
            #endregion
            
            return C;
        }

## Send and run jobs

In [ ]:
int[] MpiNUMs = new int[] {2};
//int[] MpiNUMs = new int[] {1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12};

In [ ]:
foreach(int MpiNUM in MpiNUMs){
    var C_CTRLFILE = Aland(2, 5, MpiNUM);//Create control file.
    var JobLocal = C_CTRLFILE.CreateJob();
    JobLocal.NumberOfMPIProcs = MpiNUM;
    //JobLocal.NumberOfThreads = 1;
    JobLocal.Activate();
    JobLocal.ShowOutput();
    }

In [13]:
//var C_CTRLFILE = Aland(2, 5, MpiNUM);//Create control file.

In [13]:
//var JobLocal = C_CTRLFILE.CreateJob();

In [14]:
//JobLocal.Status

PreActivation

In [15]:
//SetDefaultQueue("MSHPC-AllNodes")

In [16]:
//JobLocal.NumberOfMPIProcs = MpiNUM;
//JobLocal.NumberOfThreads = 4;
//JobLocal.Activate();
//JobLocal.ShowOutput();
//BoSSSshell.WorkflowMgm.BlockUntilAllJobsTerminate(12*60*60);

Deploying job Droplet-LikeEL-AMR5-old-MPI5-AMRMOD ... 
Opening existing database 'C:\Users\miao\Documents\Database\Droplet-EE-LikeEL'.
Set Database: { Session Count = 29; Grid Count = 316; Path = \\dc3\userspace\miao\cluster\Droplet-EE-LikeEL }
Grid is not in database yet...
Grid successfully saved: ea5cf714-28d1-4e0c-bff3-a3853fa80e54


Deploying executables and additional files ...
Deployment directory: \\dc3\userspace\miao\cluster\Droplet-EE-LikeEL-ZwoLevelSetSolver2024Apr12_120303.353349
copied 46 files.
   written file: control.obj
   copied 'win\amd64' runtime.
deployment finished.

Starting external console ...
(You may close the new window at any time, the job will continue.)


In [17]:
//JobLocal.Stdout

In [8]:
wmg.Sessions

#0: Droplet-EE-LikeEL	Droplet-LikeEL-AMR5-Merged-MPI2-AMRMODDebug*	04/25/2024 11:49:51	404f5bf8...
#1: Droplet-EE-LikeEL	Droplet-LikeEL-AMR5-Merged-MPI3-AMRMODDebug*	04/24/2024 16:36:43	1c74ded2...
#2: Droplet-EE-LikeEL	Droplet-LikeEL-AMR5-Merged-MPI3-AMRMODDebug*	04/24/2024 16:25:59	b60bc0ff...
#3: Droplet-EE-LikeEL	Droplet-LikeEL-AMR5-Merged-MPI3-AMRMODDebug*	04/24/2024 15:15:23	9d82e29a...
#4: Droplet-EE-LikeEL	Droplet-LikeEL-AMR5-Merged-MPI3-AMRMODDebug	04/24/2024 15:01:53	642137e8...
#5: Droplet-EE-LikeEL	Droplet-LikeEL-AMR5-Merged-MPI12-AMRMODDebug	04/24/2024 14:44:34	9499653d...
#6: Droplet-EE-LikeEL	Droplet-LikeEL-AMR5-Merged-MPI10-AMRMODDebug	04/24/2024 14:43:18	0522881c...
#7: Droplet-EE-LikeEL	Droplet-LikeEL-AMR5-Merged-MPI11-AMRMODDebug	04/24/2024 14:43:56	2189341a...
#8: Droplet-EE-LikeEL	Droplet-LikeEL-AMR5-Merged-MPI7-AMRMODDebug	04/24/2024 14:41:31	b7ec3afb...
#9: Droplet-EE-LikeEL	Droplet-LikeEL-AMR5-Merged-MPI8-AMRMODDebug	04/24/2024 14:42:04	2f85f08f...
#10: Droplet-

In [9]:
wmg.Sessions[0].Timesteps.Count

1143

In [ ]:
for (int i = 0; i < 1; i++){
    var outPath = wmg.Sessions[i].Timesteps.Every(1).Skip(0).Export().WithSupersampling(3).Do();
}

## Post processing

In [ ]:
//var f = databases.Pick(0).Sessions.Pick(0).Timesteps.Pick(1).GetField("Phi");

In [ ]:
//var v = databases.Pick(0).Sessions.Pick(0).Timesteps.Pick(5).GetField("VelocityX");

In [8]:
databases[0].Sessions

#0: Droplet-EE-LikeEL	Droplet-LikeEL-AMR5-Merged-MPI2-AMRMODDebug*	04/25/2024 11:44:13	94226f00...
#1: Droplet-EE-LikeEL	Droplet-LikeEL-AMR5-Merged-MPI2-AMRMODDebug	04/25/2024 11:34:05	f074503a...
#2: Droplet-EE-LikeEL	Droplet-LikeEL-AMR5-Merged-MPI4-AMRMODDebug*	04/24/2024 21:59:57	b1347d14...
#3: Droplet-EE-LikeEL	Droplet-LikeEL-AMR5-Merged-MPI4-AMRMODDebug*	04/24/2024 21:52:39	84ffbfc4...
#4: Droplet-EE-LikeEL	Droplet-LikeEL-AMR5-Merged-MPI8-AMRMODDebug*	04/24/2024 21:46:44	aecc55cb...
#5: Droplet-EE-LikeEL	Droplet-LikeEL-AMR5-Merged-MPI4-AMRMODDebug*	04/24/2024 21:26:49	4b6aa3f8...
#6: Droplet-EE-LikeEL	Droplet-LikeEL-AMR5-Merged-MPI4-AMRMODDebug*	04/24/2024 21:22:40	e7625dd0...
#7: Droplet-EE-LikeEL	Droplet-LikeEL-AMR5-Merged-MPI4-AMRMODDebug*	04/24/2024 21:11:49	c79bedc8...
#8: Droplet-EE-LikeEL	Droplet-LikeEL-AMR5-Merged-MPI4-AMRMODDebug*	04/24/2024 21:03:59	d9d81ca2...
#9: Droplet-EE-LikeEL	Droplet-LikeEL-AMR5-Merged-MPI4-AMRMODDebug*	04/24/2024 16:54:16	b3b59016...
#10: Drople

In [11]:
var session = databases[0].Sessions[0];

In [12]:
session.Timesteps.Count

1143

In [16]:
    var DisplacementY = session.Timesteps.ToDataSet(
            t => t.PhysicalTime,
            t => t.Fields.Find("DisplacementY").,
            t => "DisplacementY"
    )
    // Get VelocityX in final state. 
    double min; double max;
    VelX.GetExtremalValues(out min, out max); //Get max value of VelocityX


(3,49): error CS1001: Identifier expected

(5,6): error CS1002: ; expected



Error: compilation error

In [ ]:
    var VelX = myDb.Sessions.First().Timesteps.Last().Fields.Where(f=>f.Identification == "VelocityX").Single();
    // Get VelocityX in final state. 
    double min; double max;
    VelX.GetExtremalValues(out min, out max); //Get max value of VelocityX

In [19]:
var mydataset = session.Timesteps.ToDataSet(
        t => t.PhysicalTime,
        t => t.Fields.Where(f=>f.Identification == "VelocityX"),
        t => "DisplacementY"
        );


(3,14): error CS0029: Cannot implicitly convert type 'System.Collections.Generic.IEnumerable<BoSSS.Foundation.DGField>' to 'double'

(3,14): error CS1662: Cannot convert lambda expression to intended delegate type because some of the return types in the block are not implicitly convertible to the delegate return type



Error: compilation error

In [20]:
var mydataset = session.Timesteps.ToDataSet(
        t => t.PhysicalTime,
        t => t.Fields.Find("DisplacementY").,
        t => "DisplacementY"
        );


(3,45): error CS1001: Identifier expected



Error: compilation error

In [13]:
var mydataset = session.Timesteps.ToDataSet(
        t => t.PhysicalTime,
        t => t.Fields.Find("VelocityY").ProbeAt(0.405, 0),
        t => "VelocityY"
        );

In [11]:
//mydataset.SaveTextFileToPublish(@"C:\tmp\VelocityY.txt");

In [14]:
mydataset.PlotNow()

Using gnuplot: C:\Program Files (x86)\FDY\BoSSS\bin\native\win\gnuplot-gp510-20160418-win32-mingw\gnuplot\bin\gnuplot.exe


<?xml version="1.0" encoding="utf-8" standalone="no"?>
<!DOCTYPE svg PUBLIC "-//W3C//DTD SVG 1.1//EN"
 "http://www.w3.org/Graphics/SVG/1.1/DTD/svg11.dtd">
 

 Gnuplot 
 Produced by GNUPLOT 5.1 patchlevel 0 

 

 
 

 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 0 
 
 
 
 
 2 
 
 
 
 
 4 
 
 
 
 
 6 
 
 
 
 
 8 
 
 
 
 
 10 
 
 
 
 
 12 
 
 
 
 
 0 
 
 
 
 
 0.001 
 
 
 
 
 0.002 
 
 
 
 
 0.003 
 
 
 
 
 0.004 
 
 
 
 
 0.005 
 
 
 
 
 0.006 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 VelocityY 
 
 
 VelocityY 
 
 
 
	<path stroke='rgb( 0, 0, 0)' d='M577.4,57.1 L630.8,57.1 M45.6,564.0 L46.2,77.3 L46.7,37.3 L47.3,42.7 L47.9,45.7 L48.4,43.0
 L49.0,45.1 L49.6,47.1 L50.1,49.2 L50.7,51.2 L51.2,53.2 L51.8,55.2 L52.4,57.2 L52.9,59.2
 L53.5,61.2 L54.1,63.2 L54.6,65.2 L55.2,65.7 L55.8,67.7 L56.3,69.9 L56.9,72.1 L57.5,74.3
 L58.0,76.4 L58.6,78.6 L59.1,80.7 L59.7,82.8 L60.3,81.3 L60.8,83.4 L61.4,85.4 L62.0,87.5
 L62.5,89.5 L63.1,91.5 L63.7,93.5 L64.2,95.6 L64.8,97.6 L65.4,99.5 L65.9,101.5 L66.5,103.5
 L67.1,105.5 L67.6,107.4 L68.2,109.4 L68.7,111.3 L69.3,113.2 L69.9,115.1 L70.4,117.1 L71.0,119.0
 L71.6,114.6 L72.1,116.4 L72.7,118.2 L73.3,120.0 L73.8,121.8 L74.4,123.6 L75.0,125.4 L75.5,127.2
 L76.1,127.4 L76.7,129.1 L77.2,130.9 L77.8,132.6 L78.3,134.4 L78.9,136.1 L79.5,137.8 L80.0,139.5
 L80.6,141.2 L81.2,142.9 L81.7,144.6 L82.3,146.2 L82.9,147.9 L83.4,149.6 L84.0,151.2 L84.6,152.9
 L85.1,154.5 L85.7,156.2 L86.2,157.8 L86.8,159.4 L87.4,161.0 L87.9,162.6 L88.5,168.3 L89.1,169.8
 L89.6,171.4 L90.2,172.9 L90.8,174.5 L91.3,176.0 L91.9,177.5 L92.5,179.0 L93.0,180.5 L93.6,182.0
 L94.2,183.5 L94.7,185.0 L95.3,186.5 L95.8,188.0 L96.4,189.4 L97.0,190.9 L97.5,192.4 L98.1,193.8
 L98.7,195.3 L99.2,196.7 L99.8,198.1 L100.4,199.6 L100.9,201.0 L101.5,202.4 L102.1,203.8 L102.6,205.2
 L103.2,206.6 L103.8,208.0 L104.3,209.4 L104.9,210.8 L105.4,212.2 L106.0,213.6 L106.6,214.9 L107.1,216.3
 L107.7,217.6 L108.3,219.0 L108.8,220.3 L109.4,221.7 L110.0,223.0 L110.5,224.3 L111.1,225.7 L111.7,227.0
 L112.2,228.3 L112.8,229.6 L113.3,230.9 L113.9,232.2 L114.5,233.5 L115.0,234.8 L115.6,236.1 L116.2,237.4
 L116.7,238.6 L117.3,239.9 L117.9,241.2 L118.4,242.4 L119.0,243.7 L119.6,245.0 L120.1,246.2 L120.7,247.5
 L121.3,248.7 L121.8,249.9 L122.4,251.2 L122.9,252.4 L123.5,253.6 L124.1,254.8 L124.6,256.0 L125.2,257.3
 L125.8,258.5 L126.3,259.7 L126.9,260.9 L127.5,262.0 L128.0,263.2 L128.6,264.4 L129.2,265.6 L129.7,266.8
 L130.3,268.0 L130.9,269.1 L131.4,270.3 L132.0,271.5 L132.5,272.6 L133.1,273.8 L133.7,274.9 L134.2,276.1
 L134.8,277.2 L135.4,278.3 L135.9,279.5 L136.5,280.6 L137.1,281.7 L137.6,282.9 L138.2,284.0 L138.8,285.1
 L139.3,286.2 L139.9,287.3 L140.4,288.4 L141.0,289.5 L141.6,290.6 L142.1,291.7 L142.7,292.8 L143.3,293.9
 L143.8,295.0 L144.4,296.1 L145.0,297.2 L145.5,298.2 L146.1,299.3 L146.7,300.4 L147.2,301.4 L147.8,302.5
 L148.4,303.6 L148.9,304.6 L149.5,302.2 L150.0,303.2 L150.6,304.2 L151.2,305.2 L151.7,306.2 L152.3,307.2
 L152.9,303.2 L153.4,304.1 L154.0,305.1 L154.6,306.1 L155.1,307.1 L155.7,308.0 L156.3,309.0 L156.8,310.0
 L157.4,310.9 L157.9,311.9 L158.5,312.9 L159.1,313.8 L159.6,314.8 L160.2,315.7 L160.8,316.6 L161.3,317.6
 L161.9,318.5 L162.5,319.4 L163.0,320.4 L163.6,321.3 L164.2,322.2 L164.7,323.1 L165.3,324.1 L165.9,325.0
 L166.4,325.9 L167.0,326.8 L167.5,327.7 L168.1,328.6 L168.7,329.5 L169.2,330.4 L169.8,331.3 L170.4,332.2
 L170.9,333.1 L171.5,334.0 L172.1,334.8 L172.6,335.7 L173.2,336.6 L173.8,337.5 L174.3,338.3 L174.9,339.2
 L175.5,340.1 L176.0,340.9 L176.6,341.8 L177.1,342.6 L177.7,343.5 L178.3,344.3 L178.8,345.2 L179.4,346.0
 L180.0,346.8 L180.5,347.7 L181.1,348.5 L181.7,349.3 L182.2,350.2 L182.8,351.0 L183.4,351.8 L183.9,352.6
 L184.5,353.4 L185.0,354.2 L185.6,355.0 L186.2,355.9 L186.7,356.7 L187.3,357.5 L187.9,358.3 L188.4,359.0
 L189.0,357.7 L189.6,358.8 L190.1,359.8 L190.7,360.8 L191.3,361.7 L191.8,362.5 L192.4,374.9 

In [ ]:
var outPath = databases[0].Sessions[1].Timesteps.Every(1).Skip(352).Export().WithSupersampling(3).Do();

In [ ]:
var DispY = session.Timesteps.Last().Fields.Where(f=>f.Identification == "DisplacementY").Single();

In [ ]:
DispY

In [15]:
var mydataset = session.Timesteps.ToDataSet(
        t => t.PhysicalTime,
        t => t.Fields.Find("Phi2").ProbeAt(0.405, 0),
        t => "DisplacementY"
        );

In [16]:
mydataset.PlotNow()

Using gnuplot: C:\Program Files (x86)\FDY\BoSSS\bin\native\win\gnuplot-gp510-20160418-win32-mingw\gnuplot\bin\gnuplot.exe


<?xml version="1.0" encoding="utf-8" standalone="no"?>
<!DOCTYPE svg PUBLIC "-//W3C//DTD SVG 1.1//EN"
 "http://www.w3.org/Graphics/SVG/1.1/DTD/svg11.dtd">
 

 Gnuplot 
 Produced by GNUPLOT 5.1 patchlevel 0 

 

 
 

 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 -0.002 
 
 
 
 
 0 
 
 
 
 
 0.002 
 
 
 
 
 0.004 
 
 
 
 
 0.006 
 
 
 
 
 0.008 
 
 
 
 
 0.01 
 
 
 
 
 0.012 
 
 
 
 
 0.014 
 
 
 
 
 0.016 
 
 
 
 
 0.018 
 
 
 
 
 0 
 
 
 
 
 0.001 
 
 
 
 
 0.002 
 
 
 
 
 0.003 
 
 
 
 
 0.004 
 
 
 
 
 0.005 
 
 
 
 
 0.006 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 DisplacementY 
 
 
 DisplacementY 
 
 
 
	<path stroke='rgb( 0, 0, 0)' d='M533.0,57.1 L586.4,57.1 M70.5,511.2 L71.0,511.2 L71.6,509.8 L72.1,508.2 L72.7,506.7 L73.2,505.2
 L73.8,503.7 L74.3,502.2 L74.9,500.7 L75.4,499.2 L75.9,497.7 L76.5,496.2 L77.0,494.7 L77.6,493.2
 L78.1,491.8 L78.7,490.3 L79.2,488.8 L79.8,487.3 L80.3,485.8 L80.8,484.3 L81.4,482.8 L81.9,481.3
 L82.5,479.8 L83.0,478.3 L83.6,476.8 L84.1,475.3 L84.7,473.8 L85.2,472.3 L85.7,470.8 L86.3,469.4
 L86.8,467.9 L87.4,466.4 L87.9,464.9 L88.5,463.5 L89.0,462.0 L89.6,460.6 L90.1,459.1 L90.7,457.7
 L91.2,456.2 L91.7,454.8 L92.3,453.3 L92.8,451.9 L93.4,450.5 L93.9,449.1 L94.5,447.6 L95.0,446.2
 L95.6,444.8 L96.1,443.4 L96.6,442.0 L97.2,440.6 L97.7,439.2 L98.3,437.8 L98.8,436.4 L99.4,435.0
 L99.9,433.6 L100.5,432.2 L101.0,430.8 L101.5,429.4 L102.1,428.1 L102.6,426.7 L103.2,425.3 L103.7,424.0
 L104.3,422.6 L104.8,421.3 L105.4,419.9 L105.9,418.6 L106.4,417.2 L107.0,415.9 L107.5,414.6 L108.1,413.3
 L108.6,411.9 L109.2,410.6 L109.7,409.3 L110.3,408.0 L110.8,406.7 L111.3,405.4 L111.9,404.1 L112.4,402.8
 L113.0,401.6 L113.5,400.3 L114.1,399.1 L114.6,397.8 L115.2,396.6 L115.7,395.3 L116.2,394.1 L116.8,392.9
 L117.3,391.6 L117.9,390.4 L118.4,389.2 L119.0,388.0 L119.5,386.8 L120.1,385.6 L120.6,384.4 L121.1,383.2
 L121.7,382.0 L122.2,380.8 L122.8,379.6 L123.3,378.4 L123.9,377.3 L124.4,376.1 L125.0,374.9 L125.5,373.8
 L126.1,372.6 L126.6,371.5 L127.1,370.3 L127.7,369.2 L128.2,368.0 L128.8,366.9 L129.3,365.8 L129.9,364.6
 L130.4,363.5 L131.0,362.4 L131.5,361.3 L132.0,360.2 L132.6,359.1 L133.1,358.0 L133.7,356.9 L134.2,355.8
 L134.8,354.7 L135.3,353.6 L135.9,352.5 L136.4,351.5 L136.9,350.4 L137.5,349.3 L138.0,348.2 L138.6,347.2
 L139.1,346.1 L139.7,345.1 L140.2,344.0 L140.8,343.0 L141.3,342.0 L141.8,340.9 L142.4,339.9 L142.9,338.9
 L143.5,337.8 L144.0,336.8 L144.6,335.8 L145.1,334.8 L145.7,333.8 L146.2,332.8 L146.7,331.8 L147.3,330.8
 L147.8,329.8 L148.4,328.8 L148.9,327.8 L149.5,326.9 L150.0,325.9 L150.6,324.9 L151.1,323.9 L151.6,323.0
 L152.2,322.0 L152.7,321.1 L153.3,320.1 L153.8,319.2 L154.4,318.2 L154.9,317.3 L155.5,316.4 L156.0,315.4
 L156.5,314.5 L157.1,313.6 L157.6,312.7 L158.2,311.7 L158.7,310.8 L159.3,309.9 L159.8,309.0 L160.4,308.1
 L160.9,307.2 L161.5,306.3 L162.0,305.5 L162.5,304.6 L163.1,303.7 L163.6,302.8 L164.2,301.9 L164.7,301.1
 L165.3,300.2 L165.8,299.3 L166.4,298.5 L166.9,297.6 L167.4,296.8 L168.0,295.9 L168.5,295.1 L169.1,294.3
 L169.6,293.4 L170.2,292.6 L170.7,291.8 L171.3,290.9 L171.8,290.1 L172.3,289.3 L172.9,288.5 L173.4,287.6
 L174.0,286.8 L174.5,286.0 L175.1,285.2 L175.6,284.4 L176.2,283.6 L176.7,282.8 L177.2,282.0 L177.8,281.2
 L178.3,280.4 L178.9,279.6 L179.4,278.8 L180.0,278.0 L180.5,277.3 L181.1,276.5 L181.6,275.7 L182.1,275.0
 L182.7,274.2 L183.2,273.4 L183.8,272.7 L184.3,271.9 L184.9,271.2 L185.4,270.4 L186.0,269.7 L186.5,268.9
 L187.0,268.2 L187.6,267.5 L188.1,266.7 L188.7,266.0 L189.2,265.3 L189.8,264.6 L190.3,263.8 L190.9,263.1
 L191.4,262.4 L191.9,261.7 L192.5,261.0 L193.0,260.3 L193.6,259.6 L194.1,258.9 L194.7,258.2 L195.2,257.5
 L195.8,256.8 L196.3,256.2 L196.9,255.5 L197.4,254.8 L197.9,254.1 L198.5,253.5 L199.0,252.8 L199.6,252.1
 L200.1,251.5 L200.7,250.8 L201.2,250.1 L201.8,249.5 L202.3,248.8 L202.8,248.2 L203.4,247.6

## Restart

In [17]:
databases[0].Sessions

#0: Droplet-EE-LikeEL	Droplet-LikeEL-AMR5-Merged-MPI2-AMRMODDebug*	04/25/2024 11:49:51	404f5bf8...
#1: Droplet-EE-LikeEL	Droplet-LikeEL-AMR5-Merged-MPI3-AMRMODDebug*	04/24/2024 16:36:43	1c74ded2...
#2: Droplet-EE-LikeEL	Droplet-LikeEL-AMR5-Merged-MPI3-AMRMODDebug*	04/24/2024 16:25:59	b60bc0ff...
#3: Droplet-EE-LikeEL	Droplet-LikeEL-AMR5-Merged-MPI3-AMRMODDebug*	04/24/2024 15:15:23	9d82e29a...
#4: Droplet-EE-LikeEL	Droplet-LikeEL-AMR5-Merged-MPI3-AMRMODDebug	04/24/2024 15:01:53	642137e8...
#5: Droplet-EE-LikeEL	Droplet-LikeEL-AMR5-Merged-MPI12-AMRMODDebug	04/24/2024 14:44:34	9499653d...
#6: Droplet-EE-LikeEL	Droplet-LikeEL-AMR5-Merged-MPI10-AMRMODDebug	04/24/2024 14:43:18	0522881c...
#7: Droplet-EE-LikeEL	Droplet-LikeEL-AMR5-Merged-MPI11-AMRMODDebug	04/24/2024 14:43:56	2189341a...
#8: Droplet-EE-LikeEL	Droplet-LikeEL-AMR5-Merged-MPI7-AMRMODDebug	04/24/2024 14:41:31	b7ec3afb...
#9: Droplet-EE-LikeEL	Droplet-LikeEL-AMR5-Merged-MPI8-AMRMODDebug	04/24/2024 14:42:04	2f85f08f...
#10: Droplet-

In [18]:
//var Sloc = databases[0].Sessions.Last();
var Sloc = databases[0].Sessions[0];
Sloc

Droplet-EE-LikeEL	Droplet-LikeEL-AMR5-Merged-MPI2-AMRMODDebug*	04/25/2024 11:49:51	404f5bf8...

In [19]:
var c2 = Sloc.CreateRestartControl() as ZLS_Control;

In [20]:
c2.GetType()

ZwoLevelSetSolver.ZLS_Control

In [21]:
c2.GridGuid

060b69c3-789a-414a-9766-4b97555af7e3

In [22]:
Sloc.Timesteps.Last().GridID

7a07c215-0139-4e2c-9652-10b5c3bb9534

In [23]:
c2.GridGuid = Sloc.Timesteps.Last().GridID;

In [24]:
c2.GridGuid

7a07c215-0139-4e2c-9652-10b5c3bb9534

In [ ]:
//c2.DynamicLoadBalancing_On = true;
//c2.DynamicLoadBalancing_Period = 10;
c2.DynamicLoadBalancing_RedistributeAtStartup = false;
//c2.GridPartType = GridPartType.METIS;

//c2.AdaptiveMeshRefinement = true;
//c2.activeAMRlevelIndicators.Add(new ContactPointRefiner { maxRefinementLevel = 5});
//c2.AMR_startUpSweeps = 5;
c2.AdaptiveMeshRefinement = false;

In [ ]:
//c2.TimeSteppingScheme = TimeSteppingScheme.BDF2;

In [ ]:
var JobLocal2 = c2.CreateJob();
JobLocal2.Status

In [ ]:
JobLocal2.NumberOfMPIProcs = 2;
JobLocal2.Activate();
JobLocal2.ShowOutput();

In [ ]:
databases[0].Sessions

In [ ]:
var outPath = databases[0].Sessions[1].Timesteps.Every(1).Skip(179).Export().WithSupersampling(3).Do();

In [ ]:
databases[0].Sessions[0].